In [1]:
from helpers import init, preprocess_data

In [2]:
SEED = 42
X_train_raw, y_train_cm, X_val_raw, y_val_cm, ROOT = init(SEED, task=3)
X_train, y_train, X_val, y_val, target_min_cm, target_range_cm, rss_scale = preprocess_data(X_train_raw, y_train_cm, X_val_raw, y_val_cm)

storage_loc = "sqlite:///trail_3_no_layernorm.db"

PyTorch: 2.11.0+cu130
Training rows: 703
Validation rows: 78
Input shape: (703, 36)
Target x range (cm): 0.0 to 280.0
Target y range (cm): 0.0 to 272.0
Using split files:
  train: train_clean_6x6_8cm.csv 703 rows
  validation: validation_clean_6x6_8cm.csv 78 rows
RSS scale: 0.8427589535713196
Target min cm: [0. 0.]
Target range cm: [280. 272.]


# Get the model

In [3]:
from vlp_hackathon.baseline_model import BaselineMLP
from vlp_hackathon.improved_model import ImprovedMLP
model_cls = ImprovedMLP

# Run a trail

In [4]:
from torch import nn
import torch
import numpy as np
from matplotlib import pyplot as plt
import copy

def train_model(model, optimizer, loss_fn, epochs, batch_size, plot: bool = False, ret_model: bool = False, lr_scheduler = None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    X_train_t = torch.from_numpy(X_train).to(device)
    y_train_t = torch.from_numpy(y_train).to(device)
    X_val_t = torch.from_numpy(X_val).to(device)
    y_val_t = torch.from_numpy(y_val).to(device)
    model = model.to(device)

    history = []
    best_model = copy.deepcopy(model)
    lowest_loss = float("inf")
    train_rng = np.random.default_rng(SEED)
    for epoch in range(epochs):
        model.train()
        permutation = train_rng.permutation(len(X_train))
        running = 0.0
        for start in range(0, len(permutation), batch_size):
            idx = permutation[start:start + batch_size]
            xb = X_train_t[idx]
            yb = y_train_t[idx]
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            if loss < lowest_loss:
                best_model = copy.deepcopy(model)
            optimizer.step()
            running += float(loss.item()) * len(idx)

        model.eval()
        with torch.no_grad():
            val_loss = float(loss_fn(model(X_val_t), y_val_t).item())
        history.append((running / len(X_train), val_loss))
        lr = None
        if lr_scheduler is not None:
            lr = lr_scheduler.get_last_lr()
            lr_scheduler.step()
        print(f"epoch={epoch + 1:02d} train_mse={history[-1][0]:.6f} val_mse={val_loss:.6f} lr={lr}")

    history = np.asarray(history, dtype=np.float32)
    if plot:
        plt.figure(figsize=(6, 4))
        plt.plot(history[:, 0], label="train")
        plt.plot(history[:, 1], label="validation")
        plt.xlabel("Epoch")
        plt.ylabel("MSE on normalized coordinates")
        plt.legend()
        plt.title("Baseline training curve")
        plt.show()

    if ret_model:
        return best_model
    return np.min(history[:, 1])

def run_trial(trial):
    epochs = trial.suggest_int("epochs", 50, 150, log=True)

    print(trial)
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    batch_size = trial.suggest_int("batch_size", 16, 4096, log=True)

    depth = trial.suggest_int("depth", 1, 6)
    layers = []
    for layer in range(depth):
        width = trial.suggest_int(f"layer_{layer}", 2, 128, log=True)
        layers.append(width)

    # Start and end are fixed
    layers = [36] + layers + [2]

    act = trial.suggest_categorical("act", ["relu", "tanh", "sigmoid", ""])
    last_act = trial.suggest_categorical("last_act", ["relu", "tanh", "sigmoid", ""])
    layer_norm = False


    model = model_cls(layers=layers, activation=act, last_act=last_act, layer_norm=layer_norm)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    lr_scheduler = None
    if trial.suggest_categorical("lr_scheduler", [True, False]):
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    return train_model(model, optimizer, loss_fn, epochs, batch_size, lr_scheduler=lr_scheduler)

In [ ]:
import optuna
from optuna.trial import TrialState

study = optuna.create_study(direction='minimize', study_name="task_3", load_if_exists=True, storage=storage_loc)
study.optimize(run_trial, n_trials=1000, show_progress_bar=True)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

[I 2026-07-09 14:13:43,095] A new study created in RDB with name: task_3


  0%|          | 0/1000 [00:00<?, ?it/s]

Sequential(
  (0): Linear(in_features=36, out_features=22, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=22, out_features=2, bias=True)
  (3): ReLU()
)
epoch=01 train_mse=0.245646 val_mse=0.241837 lr=[1.770617644655569e-05]
epoch=02 train_mse=0.244456 val_mse=0.240857 lr=[1.7697750268250867e-05]
epoch=03 train_mse=0.243305 val_mse=0.239925 lr=[1.7672487773045705e-05]
epoch=04 train_mse=0.242191 val_mse=0.239074 lr=[1.7630437049535663e-05]
epoch=05 train_mse=0.241140 val_mse=0.238219 lr=[1.7571678143662944e-05]
epoch=06 train_mse=0.240163 val_mse=0.237480 lr=[1.7496322906344512e-05]
epoch=07 train_mse=0.239228 val_mse=0.236743 lr=[1.7404514780557518e-05]
epoch=08 train_mse=0.238331 val_mse=0.236010 lr=[1.7296428528287428e-05]
epoch=09 train_mse=0.237483 val_mse=0.235390 lr=[1.7172269897858665e-05]
epoch=10 train_mse=0.236698 val_mse=0.234788 lr=[1.703227523228096e-05]
epoch=11 train_mse=0.235966 val_mse=0.234202 lr=[1.6876711019357018e-05]
epoch=12 train_mse=0.235255 val_mse=0.2

In [ ]:
study = optuna.load_study(study_name="task_1", storage=storage_loc)

In [ ]:
from plotly.io import show

fig = optuna.visualization.plot_optimization_history(study)
show(fig)

fig = optuna.visualization.plot_param_importances(study)
show(fig)

fig = optuna.visualization.plot_slice(study)
show(fig)

# Make the best model

In [ ]:
best_params = study.best_params
print(best_params)

layers = []
for layer_idx in range(best_params["depth"]):
    layer_name = f"layer_{layer_idx}"
    layer_size = best_params[layer_name]
    layers.append(layer_size)
layers = [36] + layers + [2]
best_model = model_cls(layers=layers, activation=best_params["act"], last_act=best_params["last_act"])

epochs = best_params["epochs"]
batch_size = best_params["batch_size"]
optimizer = torch.optim.Adam(best_model.parameters(), lr=best_params["lr"])
if best_params["lr_scheduler"]:
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
else:
    lr_scheduler = None
loss_fn = nn.MSELoss()
best_model = train_model(best_model, optimizer, loss_fn, epochs, batch_size, plot=True, ret_model=True, lr_scheduler=lr_scheduler)

# Make the baseline model

In [ ]:
original_model = BaselineMLP(input_features=36)
history = []
epochs = 25
batch_size = 512
optimizer = torch.optim.Adam(original_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
original_model = train_model(original_model, optimizer, loss_fn, epochs, batch_size, plot=True, ret_model=True)

# Compare performance

In [ ]:
from vlp_hackathon.metrics import euclidean_errors_cm

def eval_model(model, name: str):
    model.eval()
    with torch.no_grad():
        float_norm = model(torch.from_numpy(X_val)).numpy()
    float_pred_cm = target_min_cm + float_norm * target_range_cm
    float_errors = euclidean_errors_cm(float_pred_cm, y_val_cm)

    print(f"\n{name}:")
    print(f"Float validation mean error:   {float_errors.mean():.3f} cm")
    print(f"Float validation median error: {np.median(float_errors):.3f} cm")
    print(f"Float validation p95 error:    {np.percentile(float_errors, 95):.3f} cm")
    return float_pred_cm

errors = []
for model, name in [(original_model, "original"), (best_model, "best")]:
    errors.append(eval_model(model, name))

plt.figure(figsize=(5, 5))
n_show = min(1500, len(y_val_cm))
plt.scatter(y_val_cm[:n_show, 0], y_val_cm[:n_show, 1], s=4, alpha=0.25, label="ground truth")
plt.scatter(errors[0][:n_show, 0], errors[0][:n_show, 1], s=4, alpha=0.25, label="pred_original")
plt.scatter(errors[1][:n_show, 0], errors[1][:n_show, 1], s=4, alpha=0.25, label="pred_improved")
plt.xlabel("x (cm)")
plt.ylabel("y (cm)")
plt.legend(markerscale=3)
plt.title("Clean validation positions")
plt.show()